In [ ]:
import torch
import copy
import sys
import os
import json
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)
from model import (
    load_model, 
    load_sentence_model, 
    detect,
)
from watermark.auto_watermark import AutoWatermark
from watermark.transformers_config import TransformersConfig

from attacks import baseline
from datasets import load_dataset
import numpy as np
import nltk
nltk.download('punkt_tab')
nltk.download('reuters')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

args = {
    # 'model_name_or_path': 'facebook/opt-2.7b',
    # 'model_name_or_path': 'facebook/opt-1.3b',
#     'model_name_or_path': 'facebook/opt-125m', 
    # 'model_name_or_path': 'meta-llama/Llama-3.2-1B',
    'model_name_or_path': 'facebook/opt-6.7b',
#     'model_name_or_path': 'google/gemma-7b'
    'load_fp16' : False,
    'prompt_max_length': None, 
    'max_new_tokens': 200, 
    'generation_seed': 123, 
    'use_sampling': True, 
    'n_beams': 1, 
    'sampling_temp': 0.7, 
    'use_gpu': True, 
    'seeding_scheme': 'simple_1', 
    'delta': 3.0, 
    'ignore_repeated_bigrams': False, 
    'detection_z_threshold': 4.75, 
    'select_green_tokens': True,
    'skip_model_load': False,
    'seed_separately': True,
    'topic_token_mapping': {},
    'granularity': 'kmeans',
}

In [ ]:
c4_dataset = load_dataset('json', data_files='./c4/c4.json', split='train[:1000]')
c4_iter = iter(c4_dataset)
n = 100  # how many prompts to sample
inputs_list = []
for _ in range(n):
    entry = next(c4_iter)
    text = entry["text"]
    # first 100 words
    shortened_text = " ".join(text.split()[:100])
    inputs_list.append(shortened_text)

In [ ]:
topic_list = ["animals", "technology", "sports", "medicine"]

model, tokenizer = load_model(args)
print("Model loaded")

In [ ]:
sentence_model = load_sentence_model()
print("Sentence model loaded")

In [ ]:
with open('topic_token_mapping3.json', 'r') as file:
    topic_token_mapping = json.load(file)

args['topic_token_mapping'] = topic_token_mapping

In [ ]:
gen_new_tokens = 200
transformers_config = TransformersConfig(
    model=model,
    tokenizer=tokenizer,
    vocab_size=50272,
    device=device,
    max_new_tokens=gen_new_tokens+5,
    min_new_tokens=gen_new_tokens-5,
    do_sample=True,
    no_repeat_ngram_size=4
)
args_gen = copy.deepcopy(args)
args_gen['max_new_tokens'] = gen_new_tokens+5
args_gen['min_new_tokens'] = gen_new_tokens-5

In [ ]:
def detect_KGW(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('KGW', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_DIP(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('DIP', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_Unigram(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('Unigram', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_SynthID(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('SynthID', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_EXP(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('EXP', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_EXPEdit(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('EXPEdit', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_ITSEdit(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('ITSEdit', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

def detect_SIR(watermarked_text, transformers_config):
    myWatermark = AutoWatermark.load('SIR', transformers_config=transformers_config)
    detect_result_watermarked = myWatermark.detect_watermark(watermarked_text)
    return detect_result_watermarked

In [ ]:
schemes_detection = {
    "TBW": detect,
    "KGW": detect_KGW,
    "DIP": detect_DIP,
    "Unigram": detect_Unigram,
    "SynthID": detect_SynthID,
    "SIR": detect_SIR
}

detected_results = {}
for scheme, texts in generated_text_per_scheme.items():
    print(scheme)
    detection_function = schemes_detection.get(scheme)
    detection_results = []
    if scheme == 'TBW':
        for idx, text in enumerate(texts):
            result_formatted = detection_function(inputs_list[idx], text, args_gen, device=device, tokenizer=tokenizer, sentence_model=sentence_model)
            result = {
                'is_watermarked': result_formatted[6][1] == 'Watermarked',
                'score': float(result_formatted[3][1])
            }
            print(result)
            detection_results.append(result)
    else:
        for text in texts:
            result = detection_function(text, transformers_config)
            print(result)
            detection_results.append(result)
    detected_results[scheme] = detection_results

In [ ]:
def convert_numpy_types(obj):
    if isinstance(obj, np.bool_): 
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    elif isinstance(obj, dict):
        return {key: convert_numpy_types(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(element) for element in obj]
    else:
        return obj

In [ ]:
for key, value in detected_results.items():
    print(f"Key: {key}, Value: {len(value)}")

detected_results = convert_numpy_types(detected_results)

In [ ]:
average_summary = {}
for key, value_list in detected_results.items():
    scores = [item['score'] for item in value_list]
    
    average = np.mean(scores)
    variation = np.var(scores)
    closest_index = np.argmin([abs(score - average) for score in scores])
    closest_score = scores[closest_index]
    
    average_summary[key] = {
        'average': average,
        'variation': variation,
        'index': closest_index,
        'score': closest_score
    }
    
for key, result in average_summary.items():
    print(f"{key}: {result}")

In [ ]:
N = 20

attacker = baseline.BaselineAttack()
random_attack_scores = {scheme: [] for scheme in schemes_detection.keys()}
inferred_attack_scores = {scheme: [] for scheme in schemes_detection.keys()}

perturbation_levels = list(range(0, 51, 5))

for scheme, texts in z_score_text.items():
    detection_function = schemes_detection.get(scheme)
    original_text = texts
    num_words = len(original_text.split())
    print(scheme)

    print("Random")
    for pct in perturbation_levels:
        total_edits = int(num_words * (pct / 100.0))

        runs_scores = []
        for _ in range(N):
            insertion_edits = total_edits // 3
            deletion_edits = total_edits // 3
            substitution_edits = total_edits - insertion_edits - deletion_edits

            attacked_text = attacker.combination_modify_text(
                original_text,
                insertion_n_edits=insertion_edits,
                insertion_is_inferenced=False,
                deletion_n_edits=deletion_edits,
                deletion_is_inferenced=False,
                substitution_n_edits=substitution_edits,
                substitution_is_inferenced=False
            )

            if scheme == 'TBW':
                result_formatted = detection_function(
                    inputs_list[average_summary['TBW']['index']], 
                    attacked_text, args_gen, device=device, 
                    tokenizer=tokenizer, sentence_model=sentence_model
                )
                score = float(result_formatted[3][1])
                print(score)
                runs_scores.append(score)
            else:
                result = detection_function(attacked_text, transformers_config)
                print(result)
                runs_scores.append(result['score'])

        mean_score = np.mean(runs_scores)
        std_score = np.std(runs_scores)
        random_attack_scores[scheme].append((mean_score, std_score))

    print("Inferred")
    for pct in perturbation_levels:
        total_edits = int(num_words * (pct / 100.0))

        runs_scores = []
        for _ in range(N):
            insertion_edits = total_edits // 3
            deletion_edits = total_edits // 3
            substitution_edits = total_edits - insertion_edits - deletion_edits

            attacked_text = attacker.combination_modify_text(
                original_text,
                insertion_n_edits=insertion_edits,
                insertion_is_inferenced=True,
                deletion_n_edits=deletion_edits,
                deletion_is_inferenced=True,
                substitution_n_edits=substitution_edits,
                substitution_is_inferenced=True
            )

            if scheme == 'TBW':
                result_formatted = detection_function(
                    inputs_list[average_summary['TBW']['index']], 
                    attacked_text, args_gen, device=device, 
                    tokenizer=tokenizer, sentence_model=sentence_model
                )
                score = float(result_formatted[3][1])
                print(score)
                runs_scores.append(score)
            else:
                result = detection_function(attacked_text, transformers_config)
                print(result)
                runs_scores.append(result['score'])

        mean_score = np.mean(runs_scores)
        std_score = np.std(runs_scores)
        inferred_attack_scores[scheme].append((mean_score, std_score))

    print(f"Done with scheme {scheme}")